# End-to-End E-Commerce Sales & Customer Analytics Project
### Pipeline: Data Cleaning $\rightarrow$ EDA $\rightarrow$ Visualization $\rightarrow$ SQL Database $\rightarrow$ Business Insights

**Author:** Data Analytics Portfolio Project  
**Tools & Libraries:** Python (`pandas`, `numpy`, `matplotlib`, `seaborn`, `openpyxl`), `sqlite3`, Power BI, Excel  



## Step 1: Install & Import Required Libraries


In [ ]:
# Run this cell to ensure all required packages are installed
# !pip install pandas numpy matplotlib seaborn openpyxl

import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10



## Step 2: Data Ingestion & Initial Inspection


In [ ]:
# Load original Excel dataset
input_filename = 'ecommerce_sales_customer_analysis.xlsx'

# Check sheet names
xls = pd.ExcelFile(input_filename)
print("Available Sheets:", xls.sheet_names)

# Load primary Orders sheet
df_raw = pd.read_excel(input_filename, sheet_name='Orders')
print(f"Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()



## Step 3: Data Quality Audits & Cleaning Pipeline


In [ ]:
# 1. Check for Missing Values & Duplicates
print("Missing values per column:\n", df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print(f"Duplicate Order_IDs: {df_raw['Order_ID'].duplicated().sum()}")

# 2. Trim Whitespace on all string columns
df_clean = df_raw.copy()
string_cols = df_clean.select_dtypes(include='object').columns
for col in string_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# 3. Standardize Dates & Validate Year / Month Alignment
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'])
df_clean['Year'] = df_clean['Order_Date'].dt.year
df_clean['Month'] = df_clean['Order_Date'].dt.month
df_clean['Month_Name'] = df_clean['Order_Date'].dt.strftime('%b')

# 4. Verify Revenue & Profit Margin Formulas
expected_sales = (df_clean['Quantity'] * df_clean['Unit_Price'] * (1 - df_clean['Discount'])).round(2)
sales_mismatch = (df_clean['Sales'].round(2) != expected_sales).sum()

expected_margin = ((df_clean['Profit'] / df_clean['Sales']) * 100).round(2)
margin_mismatch = (df_clean['Profit_Margin'].round(2) != expected_margin).sum()

print(f"Sales formula discrepancies: {sales_mismatch}")
print(f"Profit Margin discrepancies: {margin_mismatch}")
print("Data validation successful!")



## Step 4: Exploratory Data Analysis (EDA) & Key Metrics


In [ ]:
# High-Level Business KPIs
total_rev = df_clean['Sales'].sum()
total_profit = df_clean['Profit'].sum()
overall_margin = (total_profit / total_rev) * 100
total_orders = df_clean['Order_ID'].nunique()
unique_customers = df_clean['Customer_ID'].nunique()
aov = total_rev / total_orders

print("========================================")
print(f"Total Revenue:       ₹{total_rev:,.2f}")
print(f"Total Net Profit:    ₹{total_profit:,.2f}")
print(f"Overall Margin:      {overall_margin:.2f}%")
print(f"Total Orders:        {total_orders:,}")
print(f"Unique Customers:    {unique_customers:,}")
print(f"Average Order Value: ₹{aov:,.2f}")
print("========================================")



## Step 5: Data Visualization & Trend Analysis


In [ ]:
# Visualization 1: Monthly Sales & Profit Trend
monthly_agg = df_clean.groupby(['Year', 'Month', 'Month_Name'], as_index=False).agg({
    'Sales': 'sum',
    'Profit': 'sum'
})
monthly_agg['Period'] = monthly_agg['Month_Name'] + " " + monthly_agg['Year'].astype(str)

plt.figure(figsize=(14, 5))
plt.plot(monthly_agg['Period'], monthly_agg['Sales'] / 1e6, marker='o', color='#1F497D', label='Revenue (₹ Millions)', linewidth=2.5)
plt.plot(monthly_agg['Period'], monthly_agg['Profit'] / 1e6, marker='s', color='#107C41', label='Net Profit (₹ Millions)', linewidth=2.5)
plt.title('Monthly Sales & Profit Performance (2025 - 2026)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Period')
plt.ylabel('₹ in Millions')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# Visualization 2: Category Revenue Breakdown & Regional Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cat_sales = df_clean.groupby('Category')['Sales'].sum().sort_values(ascending=False)
axes[0].pie(cat_sales, labels=cat_sales.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('Blues_r', len(cat_sales)))
axes[0].set_title('Revenue Contribution by Product Category', fontsize=12, fontweight='bold')

sns.barplot(data=df_clean, x='Region', y='Sales', estimator=sum, errorbar=None, palette='crest', ax=axes[1])
axes[1].set_title('Total Sales by Geographical Region', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Total Revenue (₹)')
plt.tight_layout()
plt.show()



## Step 6: Create SQLite Database & Load Cleaned Table


In [ ]:
db_path = 'ecommerce_sales.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create table schema
cursor.execute('''
CREATE TABLE IF NOT EXISTS ecommerce_orders (
    order_id        TEXT PRIMARY KEY,
    order_date      TEXT NOT NULL,
    customer_id     TEXT NOT NULL,
    product_id      TEXT NOT NULL,
    category        TEXT NOT NULL,
    sub_category    TEXT NOT NULL,
    product_name    TEXT NOT NULL,
    quantity        INTEGER NOT NULL,
    unit_price      REAL NOT NULL,
    discount        REAL DEFAULT 0.0,
    sales           REAL NOT NULL,
    profit          REAL NOT NULL,
    payment_mode    TEXT NOT NULL,
    city            TEXT NOT NULL,
    state           TEXT NOT NULL,
    region          TEXT NOT NULL,
    customer_type   TEXT NOT NULL,
    shipping_days   INTEGER NOT NULL,
    order_year      INTEGER NOT NULL,
    order_month     INTEGER NOT NULL,
    month_name      TEXT NOT NULL,
    profit_margin   REAL NOT NULL
);
''')

# Prepare DataFrame for SQL insertion (renaming columns if desired)
df_sql = df_clean.rename(columns={
    'Order_ID': 'order_id',
    'Order_Date': 'order_date',
    'Customer_ID': 'customer_id',
    'Product_ID': 'product_id',
    'Category': 'category',
    'Sub_Category': 'sub_category',
    'Product_Name': 'product_name',
    'Quantity': 'quantity',
    'Unit_Price': 'unit_price',
    'Discount': 'discount',
    'Sales': 'sales',
    'Profit': 'profit',
    'Payment_Mode': 'payment_mode',
    'City': 'city',
    'State': 'state',
    'Region': 'region',
    'Customer_Type': 'customer_type',
    'Shipping_Days': 'shipping_days',
    'Year': 'order_year',
    'Month': 'order_month',
    'Month_Name': 'month_name',
    'Profit_Margin': 'profit_margin'
})
df_sql['order_date'] = df_sql['order_date'].dt.strftime('%Y-%m-%d')

df_sql.to_sql('ecommerce_orders', conn, if_exists='replace', index=False)
conn.commit()

# Verify loaded records
cursor.execute('SELECT COUNT(*) FROM ecommerce_orders')
print(f"Records successfully loaded into SQLite DB: {cursor.fetchone()[0]}")



## Step 7: Execute Core Business Queries in Python via SQL


In [ ]:
# Query 1: Top 5 Best-Selling Products by Revenue
q1 = '''
SELECT 
    product_name,
    category,
    SUM(quantity) AS units_sold,
    ROUND(SUM(sales), 2) AS total_revenue,
    ROUND(SUM(profit), 2) AS net_profit,
    ROUND((SUM(profit) / SUM(sales)) * 100, 2) AS margin_pct
FROM ecommerce_orders
GROUP BY product_name, category
ORDER BY total_revenue DESC
LIMIT 5;
'''
pd.read_sql_query(q1, conn)



In [ ]:
# Query 2: Customer Cohort Performance (New vs Returning)
q2 = '''
SELECT 
    customer_type,
    COUNT(DISTINCT customer_id) AS total_customers,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(AVG(sales), 2) AS avg_order_value,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND((SUM(profit) / SUM(sales)) * 100, 2) AS profit_margin_pct
FROM ecommerce_orders
GROUP BY customer_type;
'''
pd.read_sql_query(q2, conn)



In [ ]:
# Query 3: Payment Mode Popularity & Discount Sensitivity
q3 = '''
SELECT 
    payment_mode,
    COUNT(order_id) AS transaction_count,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(AVG(sales), 2) AS avg_order_value,
    ROUND(AVG(discount) * 100, 2) AS avg_discount_pct
FROM ecommerce_orders
GROUP BY payment_mode
ORDER BY transaction_count DESC;
'''
pd.read_sql_query(q3, conn)



## Step 8: Export Cleaned Excel with Professional Formatting


In [ ]:
output_excel = 'ecommerce_sales_cleaned_formatted.xlsx'

# Using ExcelWriter with openpyxl engine
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    df_clean.to_excel(writer, sheet_name='Orders', index=False)
    
    # Generate and export quick executive summary sheets
    cat_summary = df_clean.groupby('Category').agg(
        Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Orders=('Order_ID', 'count')
    ).reset_index()
    cat_summary.to_excel(writer, sheet_name='Category_Summary', index=False)

    reg_summary = df_clean.groupby('Region').agg(
        Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Orders=('Order_ID', 'count'),
        Customers=('Customer_ID', 'nunique')
    ).reset_index()
    reg_summary.to_excel(writer, sheet_name='Region_Summary', index=False)

print(f"Cleaned dataset exported to: {output_excel}")
conn.close()



## Step 9: Power BI DAX Measures & Final Deliverables

Create these measures inside your Power BI model for dashboard integration:

```dax
Total Sales = SUM(Orders[Sales])
Total Profit = SUM(Orders[Profit])
Profit Margin % = DIVIDE([Total Profit], [Total Sales], 0)
Total Orders = DISTINCTCOUNT(Orders[Order_ID])
AOV = DIVIDE([Total Sales], [Total Orders], 0)
YoY Sales Growth % = 
VAR CurrentSales = [Total Sales]
VAR PriorSales = CALCULATE([Total Sales], SAMEPERIODLASTYEAR(Orders[Order_Date]))
RETURN DIVIDE(CurrentSales - PriorSales, PriorSales, 0)
```

